# Una solución empresarial completa

## Ahora llevaremos nuestro proyecto del Día 1 al siguiente nivel

### DESAFÍO EMPRESARIAL:

Crear un producto que construya un folleto para una empresa para ser usado por clientes potenciales, inversores y posibles reclutas.

Se nos proporcionará un nombre de empresa y su sitio web principal.

Ver el final de este notebook para ejemplos de aplicaciones empresariales del mundo real.

Y recuerda: ¡siempre estoy disponible si tienes problemas o ideas! Por favor, no dudes en contactarme.


In [1]:
# imports
# Si estos fallan, por favor verifica que estés ejecutando desde un entorno "activado" con (llms) en el prompt de comandos

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [2]:
load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith(('AIz', 'AQ.')) and len(api_key) > 10:
    print("La API key se ve bien hasta ahora")
else:
    print("¿Podría haber un problema con tu API key? ¡Por favor visita el notebook de solución de problemas!")

MODEL = 'gemini-3.1-flash-lite'
gemini = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


La API key se ve bien hasta ahora


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## Primer paso: Que Gemini identifique qué enlaces son relevantes

### Usa una llamada a gemini-3.1-flash-lite para leer los enlaces en una página web, y responde en JSON estructurado.
Debe decidir qué enlaces son relevantes, y reemplazar enlaces relativos como "/about" con "https://empresa.com/about".
Usaremos "one-shot prompting" en el que proporcionamos un ejemplo de cómo debería responder en el prompt.

Este es un excelente caso de uso para un LLM, porque requiere una comprensión matizada. ¡Imagina intentar codificar esto sin LLMs analizando la página web — sería muy difícil!

Nota al margen: hay una técnica más avanzada llamada "Structured Outputs" en la que requerimos que el modelo responda de acuerdo con una especificación. Cubrimos esta técnica en la Semana 8 durante nuestro proyecto autónomo de Agentic AI.


In [4]:
link_system_prompt = """
Se te proporciona una lista de enlaces encontrados en una página web.
Puedes decidir cuáles de los enlaces serían más relevantes para incluir en un folleto sobre la empresa,
como enlaces a una página Acerca de, página de la empresa, o páginas de Carreras/Empleos.
Debes responder en JSON como en este ejemplo:

{
    "links": [
        {"type": "página acerca de", "url": "https://full.url/goes/here/about"},
        {"type": "página de carreras", "url": "https://another.full.url/careers"}
    ]
}
"""


In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Aquí está la lista de enlaces en el sitio web {url} -
Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa,
responde con la URL https completa en formato JSON.
No incluyas Términos de Servicio, Privacidad, enlaces de correo electrónico.

Enlaces (algunos podrían ser enlaces relativos):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


    Aquí está la lista de enlaces en el sitio web https://edwarddonner.com -
    Por favor, decide cuáles de estos son enlaces web relevantes para un folleto sobre la empresa,
    responde con la URL https completa en formato JSON.
    No incluyas Términos de Servicio, Privacidad, enlaces de correo electrónico.

    Enlaces (algunos podrían ser enlaces relativos):

    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai

In [8]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    


In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'página acerca de',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'página de currículum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'perfil profesional',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [10]:
def select_relevant_links(url):
    print(f"Seleccionando enlaces relevantes para {url} llamando a {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Encontrados {len(links['links'])} enlaces relevantes")
    return links


In [11]:
select_relevant_links("https://edwarddonner.com")

Seleccionando enlaces relevantes para https://edwarddonner.com llamando a gemini-3.1-flash-lite
Encontrados 4 enlaces relevantes


{'links': [{'type': 'página acerca de',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'página de currículum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'página de inicio', 'url': 'https://edwarddonner.com/'},
  {'type': 'perfil profesional',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Seleccionando enlaces relevantes para https://huggingface.co llamando a gemini-3.1-flash-lite
Encontrados 6 enlaces relevantes


{'links': [{'type': 'página acerca de',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'página de empresa', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'página de carreras',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'página de marca', 'url': 'https://huggingface.co/brand'},
  {'type': 'página de blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'página de documentación', 'url': 'https://huggingface.co/docs'}]}

## Segundo paso: ¡hacer el folleto!

Reunir todos los detalles en otro prompt a Gemini


In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Seleccionando enlaces relevantes para https://edwarddonner.com llamando a gemini-3.1-flash-lite
Encontrados 4 enlaces relevantes
## Landing Page:

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated cour

In [18]:
brochure_system_prompt = """
Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa
y crea un folleto corto sobre la empresa para clientes potenciales, inversores y reclutas.
Responde en markdown sin bloques de código.
Incluye detalles de la cultura empresarial, clientes y carreras/empleos si tienes la información.
"""

# O descomenta las líneas de abajo para un folleto más humorístico - esto demuestra lo fácil que es incorporar 'tono':

# brochure_system_prompt = """
# Eres un asistente que analiza el contenido de varias páginas relevantes del sitio web de una empresa
# y crea un folleto corto, humorístico, entretenido y ingenioso sobre la empresa para clientes potenciales, inversores y reclutas.
# Responde en markdown sin bloques de código.
# Incluye detalles de la cultura empresarial, clientes y carreras/empleos si tienes la información.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
Estás mirando una empresa llamada: {company_name}
Aquí está el contenido de su página de inicio y otras páginas relevantes;
usa esta información para construir un folleto corto de la empresa en markdown sin bloques de código.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncar si hay más de 5.000 caracteres
    return user_prompt


In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Seleccionando enlaces relevantes para https://huggingface.co llamando a gemini-3.1-flash-lite
Encontrados 5 enlaces relevantes


"\nEstás mirando una empresa llamada: HuggingFace\nAquí está el contenido de su página de inicio y otras páginas relevantes;\nusa esta información para construir un folleto corto de la empresa en markdown sin bloques de código.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nCloudflare/clef\nUpdated\n3 days ago\n•\n4.21k\n•\n1.16k\nconvaiinnovations/laya\nUpdated\n1 day ago\n•\n

In [21]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))


In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Seleccionando enlaces relevantes para https://huggingface.co llamando a gemini-3.1-flash-lite
Encontrados 7 enlaces relevantes


# Hugging Face: El motor de la colaboración en Inteligencia Artificial

### Sobre la empresa
Hugging Face es la plataforma líder mundial para la comunidad de aprendizaje automático (Machine Learning). Definida como "el hogar del Machine Learning", la misión de la empresa es democratizar la IA mediante la creación de un ecosistema abierto, colaborativo y accesible donde investigadores, desarrolladores y empresas pueden crear, descubrir y compartir soluciones tecnológicas de vanguardia.

### ¿Qué ofrecemos?
La plataforma sirve como un centro neurálgico donde la comunidad interactúa a través de:
* **Modelos:** Acceso a más de 2 millones de modelos pre-entrenados para diversas tareas.
* **Datasets:** Acceso a más de 500,000 conjuntos de datos para el entrenamiento y ajuste de modelos.
* **Spaces:** Un entorno interactivo para alojar, explorar y demostrar aplicaciones de IA de forma rápida.
* **Soluciones Enterprise:** Herramientas diseñadas para equipos profesionales, incluyendo soporte técnico, puntos de conexión de inferencia (Inference Endpoints) y almacenamiento seguro (Buckets).

### Cultura y Comunidad
Hugging Face se distingue por ser una plataforma impulsada por la comunidad. La cultura de la empresa se basa en la **apertura y el intercambio**. Fomentamos un entorno donde la innovación ocurre de manera colectiva, facilitando que personas de todo el mundo colaboren en proyectos desde el desarrollo técnico hasta la resolución de problemas complejos. Si te apasiona el código abierto y el futuro de la inteligencia artificial, este es el lugar donde el trabajo colaborativo define el estándar de la industria.

### Clientes y Usuarios
Nuestra comunidad es vasta y diversa:
* **Investigadores y Científicos de Datos:** Que utilizan la plataforma para publicar sus descubrimientos y acceder a los últimos avances.
* **Desarrolladores:** Que integran modelos y datasets en sus propias aplicaciones.
* **Empresas:** Desde startups hasta grandes corporaciones que utilizan los planes "Team & Enterprise" para escalar sus infraestructuras de IA, garantizar la seguridad de sus datos y contar con soporte especializado de nivel experto.

### Carreras y Talento
En Hugging Face, buscamos personas apasionadas por el impacto global de la IA. Trabajar aquí significa formar parte de una infraestructura esencial que está dando forma al futuro de la tecnología. Buscamos talento con visión técnica, espíritu colaborativo y compromiso con el ecosistema de código abierto. Si buscas un lugar donde tu trabajo llegue a millones de desarrolladores y acelere el progreso de la IA, te invitamos a ser parte de nuestra comunidad.

---
**¿Quieres saber más?**
Explora nuestra plataforma en [huggingface.co](https://huggingface.co), únete a nuestro Discord, participa en nuestros foros o revisa nuestro GitHub para empezar a construir el futuro de la IA hoy mismo.

## Finalmente - una mejora menor

Con un pequeño ajuste, podemos cambiar esto de modo que los resultados fluyan desde Gemini,
con la familiar animación de máquina de escribir


In [23]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Seleccionando enlaces relevantes para https://huggingface.co llamando a gemini-3.1-flash-lite
Encontrados 6 enlaces relevantes


# Hugging Face: Construyendo el Futuro de la IA

### Sobre la Empresa
Hugging Face es la plataforma de colaboración líder para la comunidad de aprendizaje automático (Machine Learning). Conocida como "el hogar del Machine Learning", la empresa facilita el descubrimiento, la creación y el intercambio de modelos, conjuntos de datos (datasets) y aplicaciones de inteligencia artificial. Su misión es democratizar la IA mediante una infraestructura abierta y colaborativa.

### Cultura Empresarial
La cultura de Hugging Face está profundamente arraigada en el espíritu del **código abierto y la comunidad**. La empresa opera como un nodo central donde investigadores, desarrolladores y empresas convergen para acelerar el progreso tecnológico. La transparencia, la experimentación y el intercambio de conocimientos son los pilares que sostienen su ecosistema, fomentando un entorno donde la innovación es colectiva y accesible.

### Clientes y Soluciones
Hugging Face ofrece soluciones diseñadas tanto para individuos como para equipos corporativos de gran escala:
*   **Comunidad Global:** Millones de usuarios acceden diariamente a más de 2 millones de modelos y 500,000 conjuntos de datos.
*   **Enterprise:** La plataforma proporciona herramientas robustas para empresas, incluyendo:
    *   **Inference Endpoints:** Soluciones para desplegar modelos de manera eficiente.
    *   **Enterprise Support:** Soporte técnico especializado para organizaciones que integran IA en sus procesos de negocio.
    *   **Almacenamiento y Cómputo:** Infraestructura escalable como Storage Buckets e Inference Providers para gestionar grandes cargas de trabajo de ML.
*   **Hugging Face PRO:** Membresías para usuarios individuales que buscan capacidades adicionales en su flujo de trabajo.

### Carreras y Oportunidades
Trabajar en Hugging Face es unirse a un equipo que está definiendo el estándar de la industria en IA. La empresa busca personas apasionadas por el impacto global, la ingeniería de alta complejidad y el fomento de ecosistemas abiertos. Al ser una organización centrada en la comunidad, se valora la capacidad de construir puentes entre la investigación académica y la implementación práctica en el mundo real.

Si eres un ingeniero, investigador o profesional del ámbito tecnológico interesado en contribuir a una plataforma que impulsa las tendencias de IA más importantes de la actualidad, Hugging Face ofrece un entorno dinámico, descentralizado y de vanguardia.

---
**¿Quieres ser parte del futuro de la IA?**
Explora los recursos, únete a la comunidad en Discord o revisa nuestras herramientas en [huggingface.co](https://huggingface.co).

In [ ]:
# Intenta cambiar el system prompt a la versión humorística cuando hagas el folleto para Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Aplicaciones empresariales</h2>
            <span style="color:#181;">En este ejercicio extendimos el código del Día 1 para hacer múltiples llamadas a LLM, y generar un documento.

Este es quizá el primer ejemplo de patrones de diseño de Agentic AI, ya que combinamos múltiples llamadas a LLMs. Esto aparecerá más en la Semana 2, y luego regresaremos a Agentic AI de una manera importante en la Semana 8 cuando construyamos una solución de Agente completamente autónoma.

La generación de contenido de esta manera es uno de los casos de uso muy más comunes. Al igual que con la resumición, esto se puede aplicar a cualquier vertical empresarial. Escribe contenido de marketing, genera un tutorial de producto a partir de una especificación, crea contenido de correo electrónico personalizado, y mucho más. Explora cómo puedes aplicar la generación de contenido a tu negocio, e intenta hacer un prototipo de prueba de concepto por ti mismo. ¡Mira lo que otros estudiantes han hecho en la carpeta community-contributions — tantos proyectos valiosos — es salvaje!</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Antes de pasar a la Semana 2 (que es un montón de diversión)</h2>
            <span style="color:#900;">Por favor, ve al notebook de EJERCICIO de week1 para tu desafío al final de la semana 1. Esto te dará una práctica esencial trabajando con APIs de Frontera, y te preparará bien para la Semana 2.</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Un recordatorio sobre 3 recursos útiles</h2>
            <span style="color:#f71;">1. Los recursos del curso están disponibles <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">aquí.</a><br/>
            2. ¡Estoy en LinkedIn <a href="https://www.linkedin.com/in/eddonner/">aquí</a> y me encanta conectar con personas que toman el curso!<br/>
            3. Estoy probando X/Twitter y estoy en <a href="https://x.com/edwarddonner">@edwarddonner</a> y esperando que la gente me enseñe cómo se hace..  
            </span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">¡Por fin! Tengo una solicitud especial para ti</h2>
            <span style="color:#090;">
                Mi editor me dice que hace una DIFERENCIA MASIVA cuando los estudiantes califican este curso en Udemy - es una de las formas principales en que Udemy decide si mostrarlo a otros. Si puedes tomarte un minuto para calificarlo, ¡te estaría muy agradecido! Y de todos modos - siempre por favor comunícate conmigo en ed@edwarddonner.com si puedo ayudarte en algún punto.
            </span>
        </td>
    </tr>
</table>
